# Transform Vehicles — Capa Silver

## Hallazgos del diagnóstico de calidad
El ETL original tenía un join exacto `join_make + join_model` que producía sólo **0.34 % de match** (721 / 210 165 registros).

### Causas raíz identificadas

| # | Problema | Impacto en las tablas golden |
|---|---|---|
| 1 | **Nombres de modelo incompatibles**: specs tiene variantes completas ("MODEL 3 LONG RANGE AWD (HIGHLAND)") y population el nombre base ("MODEL 3") | `avg_specs_range_km` = NaN en toda la tabla `golden_make_range_price_ratio` |
| 2 | **16 marcas sin cobertura en specs**: CHEVROLET, RIVIAN, ALFA ROMEO, NISSAN (LEAF), TESLA (LEAF no en specs 2025)… | Esas marcas nunca tendrán `specs_range_km` |
| 3 | **`base_msrp` inútil como precio real**: 98.4 % de los registros tiene MSRP = 0 (campo no reportado en el DOL de WA) | `avg_price` y `range_per_price_ratio` no son fiables |
| 4 | **`electric_range` en MILLAS** (EPA, desde el DOL de WA), `range_km` en KILÓMETROS (ev-database.org) | Comparación directa es incorrecta — hay que convertir |
| 5 | **Datos sesgados geográficamente**: 99.8 % de registros son de WA | `golden_state_distribution` no representa EEUU |

### Solución implementada en este notebook
- Join mejorado: `join_make` exacto + modelo population como **prefijo** del modelo specs  
  → toma los specs del primer match por mark+make y promedia cuando hay varios  
- `electric_range` se convierte a km (`electric_range_km = electric_range * 1.60934`)  
- `base_msrp` se mantiene pero se añade columna `has_valid_msrp` para filtrar análisis


In [ ]:
dbutils.widgets.removeAll()

from pyspark.sql import functions as F

In [ ]:
dbutils.widgets.text("catalogo", "catalog_smartdata_final")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink_silver", "silver")
dbutils.widgets.text("storageName", "adlssmartdata2023")
dbutils.widgets.dropdown("writeMode", "overwrite", ["overwrite", "append"])

In [ ]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink_silver = dbutils.widgets.get("esquema_sink_silver")
storage_name = dbutils.widgets.get("storageName")
write_mode = dbutils.widgets.get("writeMode")

tbl_specs_bronze = f"{catalogo}.{esquema_source}.ev_specs_2025_bronze"
tbl_population_bronze = f"{catalogo}.{esquema_source}.ev_population_bronze"

tbl_model_specs_silver = f"{catalogo}.{esquema_sink_silver}.ev_model_specs"
tbl_population_clean_silver = f"{catalogo}.{esquema_sink_silver}.ev_population_clean"
tbl_vehicle_enriched_silver = f"{catalogo}.{esquema_sink_silver}.ev_vehicle_enriched"

In [ ]:
normalize_text = lambda c: F.upper(F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))

df_specs_bronze      = spark.table(tbl_specs_bronze)
df_population_bronze = spark.table(tbl_population_bronze)

# ---------- SILVER: ev_model_specs ----------
df_specs_silver = (
    df_specs_bronze
    .dropna(how="all")
    .filter(F.col("brand").isNotNull() & F.col("model").isNotNull())
    .select(
        normalize_text("brand").alias("brand"),
        normalize_text("model").alias("model"),
        normalize_text("brand").alias("join_make"),
        normalize_text("model").alias("join_model"),
        F.col("battery_capacity_kwh"),
        F.col("range_km"),
        F.col("efficiency_wh_per_km"),
        F.col("fast_charging_power_kw_dc"),
        F.col("seats"),
        F.col("drivetrain"),
        F.col("segment"),
        F.col("car_body_type"),
        F.col("source_url"),
        F.current_timestamp().alias("updated_ts")
    )
    .dropDuplicates(["join_make", "join_model"])
)

# ---------- SILVER: ev_population_clean ----------
# Se conserva electric_range original (millas EPA) y se agrega conversión a km
df_population_silver = (
    df_population_bronze
    .dropna(how="all")
    .filter(F.col("make").isNotNull() & F.col("model").isNotNull())
    .select(
        F.col("vin_1_10"),
        normalize_text("state").alias("state"),
        normalize_text("county").alias("county"),
        normalize_text("city").alias("city"),
        F.col("postal_code"),
        F.col("model_year"),
        normalize_text("make").alias("make"),
        normalize_text("model").alias("model"),
        normalize_text("make").alias("join_make"),
        normalize_text("model").alias("join_model"),
        F.col("electric_vehicle_type"),
        F.col("cafv_eligibility"),
        F.col("electric_range"),
        F.round(F.col("electric_range") * 1.60934, 2).alias("electric_range_km"),
        F.col("base_msrp"),
        (F.col("base_msrp") > 0).alias("has_valid_msrp"),
        F.col("dol_vehicle_id"),
        F.col("vehicle_location"),
        F.col("electric_utility"),
        F.col("census_tract_2020"),
        F.current_timestamp().alias("updated_ts")
    )
)

# ---------- JOIN MEJORADO: exact + prefijo de modelo + fallback de marca ----------
#
# Problema original: join exacto (make + model) → 0.34 % de match,
# porque specs tiene variantes completas ("MODEL 3 LONG RANGE AWD")
# y population sólo el nombre base ("MODEL 3").
#
# Problema adicional: Column.startswith(Column) en PySpark no es fiable
# en todos los runtimes de Databricks → se usa SQL LIKE CONCAT(..., '%').
#
# Solución (3 niveles):
#   1. Join exacto en join_make + join_model
#   2. Join por prefijo: specs.join_model LIKE CONCAT(pop.join_model, '%')
#      → cuando hay varios specs que prefixan, se promedian los valores numéricos
#   3. Fallback por marca: promedio de todos los specs de la misma marca
#      → garantiza que cualquier marca conocida en specs tenga specs_range_km
#   El enriquecimiento final usa COALESCE(exacto, prefijo, marca)

# Nivel 2 – Specs agregados por prefijo de modelo
# NOTA: se usa F.expr con LIKE CONCAT en lugar de Column.startswith(Column)
# porque este último no es fiable con columnas dinámicas en Databricks.
df_specs_prefix = (
    df_specs_silver.alias("s")
    .join(
        df_population_silver.select("join_make", "join_model").distinct().alias("p"),
        on=(
            (F.col("s.join_make") == F.col("p.join_make")) &
            F.expr("s.join_model LIKE CONCAT(p.join_model, '%')")
        ),
        how="inner"
    )
    .groupBy(F.col("p.join_make").alias("pm"), F.col("p.join_model").alias("pmodel"))
    .agg(
        F.avg("s.range_km").alias("specs_range_km_avg"),
        F.avg("s.efficiency_wh_per_km").alias("specs_efficiency_avg"),
        F.avg("s.fast_charging_power_kw_dc").alias("specs_fast_charge_avg"),
        F.first("s.battery_capacity_kwh").alias("specs_battery_kwh"),
        F.first("s.segment").alias("specs_segment"),
        F.first("s.drivetrain").alias("specs_drivetrain"),
        F.first("s.car_body_type").alias("specs_car_body_type"),
        F.first("s.seats").alias("specs_seats"),
        F.count(F.lit(1)).alias("specs_variant_count")
    )
)

# Nivel 3 – Fallback: promedio de specs por marca
# Cubre modelos de población sin match exacto ni de prefijo (ej. BMW "330E", KIA "SOUL")
df_specs_make = (
    df_specs_silver
    .groupBy(F.col("join_make").alias("mk"))
    .agg(
        F.avg("range_km").alias("make_range_km"),
        F.avg("efficiency_wh_per_km").alias("make_efficiency"),
        F.avg("fast_charging_power_kw_dc").alias("make_fast_charge"),
        F.avg("battery_capacity_kwh").alias("make_battery_kwh"),
        F.first("segment").alias("make_segment"),
        F.first("drivetrain").alias("make_drivetrain"),
        F.first("car_body_type").alias("make_car_body_type"),
        F.first("seats").alias("make_seats"),
    )
)

df_vehicle_enriched = (
    df_population_silver.alias("p")
    .join(df_specs_silver.alias("se"), on=["join_make", "join_model"], how="left")
    .join(
        df_specs_prefix.alias("sp"),
        (F.col("p.join_make") == F.col("sp.pm")) &
        (F.col("p.join_model") == F.col("sp.pmodel")),
        how="left"
    )
    .join(
        df_specs_make.alias("sm"),
        F.col("p.join_make") == F.col("sm.mk"),
        how="left"
    )
    .select(
        F.col("p.vin_1_10"),
        F.col("p.state"),
        F.col("p.county"),
        F.col("p.city"),
        F.col("p.model_year"),
        F.col("p.make"),
        F.col("p.model"),
        F.col("p.join_make"),
        F.col("p.join_model"),
        F.col("p.electric_vehicle_type"),
        F.col("p.cafv_eligibility"),
        F.col("p.electric_range").alias("registered_electric_range"),
        F.col("p.electric_range_km").alias("registered_electric_range_km"),
        F.col("p.base_msrp"),
        F.col("p.has_valid_msrp"),
        F.col("se.brand").alias("specs_brand"),
        F.col("se.model").alias("specs_model"),
        F.coalesce(F.col("se.battery_capacity_kwh"),      F.col("sp.specs_battery_kwh"),   F.col("sm.make_battery_kwh")).alias("specs_battery_capacity_kwh"),
        F.coalesce(F.col("se.range_km").cast("double"),   F.col("sp.specs_range_km_avg"),  F.col("sm.make_range_km")).alias("specs_range_km"),
        F.coalesce(F.col("se.efficiency_wh_per_km"),      F.col("sp.specs_efficiency_avg"), F.col("sm.make_efficiency")).alias("specs_efficiency_wh_per_km"),
        F.coalesce(F.col("se.fast_charging_power_kw_dc"), F.col("sp.specs_fast_charge_avg"), F.col("sm.make_fast_charge")).alias("specs_fast_charging_power_kw_dc"),
        F.coalesce(F.col("se.drivetrain"),   F.col("sp.specs_drivetrain"),   F.col("sm.make_drivetrain")).alias("specs_drivetrain"),
        F.coalesce(F.col("se.segment"),      F.col("sp.specs_segment"),      F.col("sm.make_segment")).alias("specs_segment"),
        F.coalesce(F.col("se.car_body_type"), F.col("sp.specs_car_body_type"), F.col("sm.make_car_body_type")).alias("specs_car_body_type"),
        F.col("se.join_make").isNotNull().alias("has_exact_match"),
        F.col("sp.pm").isNotNull().alias("has_prefix_match"),
        (F.col("se.join_make").isNotNull() | F.col("sp.pm").isNotNull()).alias("has_specs_match"),
        F.col("sp.specs_variant_count"),
        F.current_timestamp().alias("updated_ts")
    )
)


In [ ]:
df_specs_silver.write.mode(write_mode).insertInto(tbl_model_specs_silver)
df_population_silver.write.mode(write_mode).option("overwriteSchema", "true").saveAsTable(tbl_population_clean_silver)
df_vehicle_enriched.write.mode(write_mode).option("overwriteSchema", "true").saveAsTable(tbl_vehicle_enriched_silver)

total         = df_vehicle_enriched.count()
exact_match   = df_vehicle_enriched.filter(F.col("has_exact_match")).count()
prefix_match  = df_vehicle_enriched.filter(F.col("has_prefix_match") & ~F.col("has_exact_match")).count()
any_match     = df_vehicle_enriched.filter(F.col("has_specs_match")).count()
no_match      = total - any_match

print(f"silver model specs     : {df_specs_silver.count():>8,}")
print(f"silver population clean: {df_population_silver.count():>8,}")
print(f"silver vehicle enriched: {total:>8,}")
print()
print(f"--- Calidad del join ---")
print(f"Match exacto (make+model) : {exact_match:>8,}  ({exact_match/total*100:.2f}%)")
print(f"Match por prefijo (nuevo) : {prefix_match:>8,}  ({prefix_match/total*100:.2f}%)")
print(f"Total con specs           : {any_match:>8,}  ({any_match/total*100:.2f}%)")
print(f"Sin match (make no en specs o modelo nuevo): {no_match:>8,}  ({no_match/total*100:.2f}%)")


In [ ]:
# Marcas con y sin cobertura en specs
df_coverage = (
    df_vehicle_enriched
    .groupBy("make")
    .agg(
        F.count(F.lit(1)).alias("total_registros"),
        F.sum(F.col("has_specs_match").cast("int")).alias("con_specs"),
    )
    .withColumn("pct_cobertura", F.round(F.col("con_specs") / F.col("total_registros") * 100, 1))
    .orderBy(F.desc("total_registros"))
)
display(df_coverage)
